# Evaluación 03 — Detección de maquinaria de construcción mediante YOLO
**Mathis Marsault · Julien Martinat · Louis Maillet**  
**Universidad Bernardo O'Higgins · Machine Learning · Dr. Eliecer Peña Ancavil**

Este notebook implementa el protocolo de las **Partes 3–10** de la plantilla oficial `ML Clase10 Caras YOLO.ipynb` entregada por el docente. Conserva las dos fases obligatorias de entrenamiento, la comparación en Validation y una sola evaluación Test.

**Dataset propuesto:** Excavators, versión 2: https://universe.roboflow.com/mohamed-sabek-6zmr6/excavators-cwlh0/dataset/2

**IMPORTANTE:** aún sin ejecutar. No hay datos ni resultados numéricos incluidos. Activar GPU T4 en Colab. El ZIP `ejemplos.zip` de la clase no es un dataset de maquinaria. Evitar fugas de datos y completar fotos/video externos para obtener la puntuación del enunciado.

**Dataset verificado (2026-10-29):** ZIP v2 original con 2 656 imágenes y 3 881 cajas. Debido a escenas de los mismos videos y 24 duplicados visuales en diferentes particiones la celda de importación prepara automáticamente una versión corregida a partir del ZIP original (**Train 2 406, Validation 144, Test 82**). El notebook registra la limpieza y guarda la partición corregida antes del entrenamiento.

## 1 — Dependencias, GPU y repositorio

In [ ]:
%pip install -q ultralytics kagglehub pyyaml pandas matplotlib pillow pytest
import os, sys, shutil, json, zipfile, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Activar GPU T4 en Entorno de ejecución > Cambiar tipo de entorno de ejecución"
print("GPU:", torch.cuda.get_device_name(0))
if not Path("/content/YOLO-Maquinaria-UBO/src/yolo_ubo.py").exists():
    subprocess.run(["git", "clone", "https://github.com/MamatorHack/YOLO-Maquinaria-UBO.git", "/content/YOLO-Maquinaria-UBO"], check=True)
ROOT = Path("/content/YOLO-Maquinaria-UBO")
# Al reabrir Colab, actualizar los scripts incluso si el repositorio ya existía.
subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(ROOT))
from src.yolo_ubo import *
%cd /content/YOLO-Maquinaria-UBO

## 2 — Configuración del experimento
**No modificar después de ver Test.** Dos etapas obligatorias: congelar backbone en transfer learning, descongelar todo en fine tuning. Estos valores son una configuración inicial adaptada a las limitaciones de Colab gratuito.

In [ ]:
SEED = 42
IMGSZ = 416
BATCH = 16  # Reducir a 8 si ocurre CUDA out of memory
TRAIN_FRACTION = 1.0  # Reducir (ej. 0.5) si el tiempo Colab no alcanza
EPOCHS_TRANSFER = 8
EPOCHS_FINETUNE = 5
BASE_WEIGHTS = "yolo26n.pt"  # alternativa permitida: yolo11n.pt
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50
CLASES = ("EXCAVATORS", "dump truck", "wheel loader")
Path("results").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)
import random
random.seed(SEED)
print("Configuración:", {k:v for k,v in globals().copy().items() if k in ("IMGSZ","BATCH","TRAIN_FRACTION","EPOCHS_TRANSFER","EPOCHS_FINETUNE","BASE_WEIGHTS","SEED")})

## 3 — Datos: ZIP YOLO, limpieza automática y preparación sin fuga evidente
La celda recupera el ZIP original **ya presente en Colab** o solicita subirlo cuando falta, aplica el script de corrección reproducible si procede y conserva Train/Validation/Test corregidos. No es necesario descargar un segundo ZIP desde ChatGPT. Revisar igualmente escenas parecidas y limitaciones del conjunto.

In [ ]:
from google.colab import files
# Se reutiliza el ZIP original ya subido a Colab; no requiere descargar un ZIP gigante desde ChatGPT.
existing = list(Path("/content").rglob("Excavators.v2-UBO-sin-fugas-evidentes.yolov8.zip"))
existing += list(Path("/content").rglob("Excavators.v2-raw-images_416by416.yolov8.zip"))
if existing:
    zip_path = existing[0]
    print("Reutilizando archivo ya presente en Colab:", zip_path)
else:
    uploaded = files.upload()
    zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
    assert len(zip_names) == 1, "Subir exactamente un ZIP YOLO"
    zip_path = Path(zip_names[0])
# Cuando se usa la exportación Roboflow original, limpiar escenas duplicadas EN COLAB.
with zipfile.ZipFile(zip_path) as z:
    already_cleaned = "LIMPIEZA_DATOS.json" in z.namelist()
if not already_cleaned:
    print("ZIP original detectado: preparando particiones sin fugas visuales evidentes...")
    from scripts.preparar_dataset_sin_fugas import prepare as limpiar_dataset
    clean_zip = Path("/content/Excavators.v2-UBO-sin-fugas-evidentes.yolov8.zip")
    limpiar_dataset(zip_path, clean_zip)
    zip_path = clean_zip
extract_dir = Path("/content/yolo_original")
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    limpieza = json.loads(z.read("LIMPIEZA_DATOS.json"))
    Path("results/dataset_cleaning.json").write_text(json.dumps(limpieza, indent=2, ensure_ascii=False))
    print("Limpieza:", limpieza.get("method", "ver LIMPIEZA_DATOS.json"))
    for member in z.infolist():
        target = (extract_dir / member.filename).resolve()
        if not target.is_relative_to(extract_dir.resolve()):
            raise ValueError("Ruta insegura dentro del ZIP")
    z.extractall(extract_dir)
original_yaml = find_dataset(extract_dir)
print("data.yaml original:\n", original_yaml.read_text()[:2500])
DATA_YAML = prepare_dataset(original_yaml, "/content/yolo_processed", CLASES)
print("Nuevo YAML:", DATA_YAML)
counts = count_dataset(DATA_YAML)
print(json.dumps(counts, indent=2, ensure_ascii=False))

## 4 — Auditoría de fuga, visualización de cajas y ejemplo de etiqueta
Un label YOLO es: `clase x_centro y_centro ancho alto`, con coordenadas normalizadas entre 0 y 1. Un SHA256 solo detecta imágenes **idénticas byte por byte**; inspeccionar manualmente escenas similares y copias transformadas.

In [ ]:
audit = check_exact_leakage(DATA_YAML)
print(json.dumps(audit, indent=2, ensure_ascii=False))
assert not audit["exact_duplicates_across_splits"], "Fuga de imágenes idénticas: corregir ANTES de entrenar"
from IPython.display import display, Image
sample_path = plot_samples(DATA_YAML, split="train", output="results/annotations_train.png")
display(Image(filename=sample_path))
first_label = next((Path("/content/yolo_processed/train/labels")).rglob("*.txt"))
print("Etiqueta de ejemplo:", first_label.name, first_label.read_text().splitlines()[:2])
print("Revisión manual pendiente: verificar fotos reales, escenas parecidas y calidad de cajas.")

## 5 — Transfer learning obligatorio
Partimos de pesos preentrenados, congelando el backbone con `freeze=10`.

In [ ]:
transfer = train_stage(BASE_WEIGHTS, DATA_YAML, "transfer_learning", EPOCHS_TRANSFER,
                       IMGSZ, BATCH, freeze=10, lr0=0.01, optimizer="auto",
                       fraction=TRAIN_FRACTION, project="/content/yolo_runs", seed=SEED)
shutil.copy2(transfer["best"], "models/transfer_learning.pt")
print(json.dumps(transfer, indent=2))
plot_training_curves(transfer["results_csv"], "results/transfer_curves.png")
display(Image(filename="results/transfer_curves.png"))

## 6 — Fine tuning obligatorio
Reanudar desde el **best.pt obtenido por nosotros**; `freeze=0`, AdamW y tasa de aprendizaje baja.

In [ ]:
fine = train_stage(transfer["best"], DATA_YAML, "fine_tuning", EPOCHS_FINETUNE,
                   IMGSZ, BATCH, freeze=0, lr0=0.0001, optimizer="AdamW",
                   fraction=TRAIN_FRACTION, project="/content/yolo_runs", seed=SEED)
shutil.copy2(fine["best"], "models/fine_tuning.pt")
print(json.dumps(fine, indent=2))
plot_training_curves(fine["results_csv"], "results/fine_curves.png")
display(Image(filename="results/fine_curves.png"))
Path("results/train_config.json").write_text(json.dumps({"transfer":transfer,"fine_tuning":fine}, indent=2))

## 7 — Validation: medir los dos modelos
Registrar precision, recall, mAP50 y mAP50-95. **Todavía no abrir Test**.

In [ ]:
val_transfer = evaluate("models/transfer_learning.pt", DATA_YAML, split="val", imgsz=IMGSZ)
val_fine = evaluate("models/fine_tuning.pt", DATA_YAML, split="val", imgsz=IMGSZ)
import pandas as pd
comparison = pd.DataFrame([{"modelo":"transfer_learning",**val_transfer},{"modelo":"fine_tuning",**val_fine}])
display(comparison)
comparison.to_csv("results/validation_comparison.csv", index=False)

## 8 — Elección ANTES de Test
Regla escrita de antemano: máximo mAP50-95 en Validation; desempate mAP50 y luego transfer learning. La elección queda guardada y no se puede sobrescribir accidentalmente.

In [ ]:
selection = select_model(val_transfer, val_fine, output="results/selection.json")
chosen_path = "models/fine_tuning.pt" if selection["selected"] == "fine_tuning" else "models/transfer_learning.pt"
shutil.copy2(chosen_path, "models/modelo_elegido.pt")
print(json.dumps(selection, indent=2))
from ultralytics import YOLO
for model_path in ["models/transfer_learning.pt","models/fine_tuning.pt","models/modelo_elegido.pt"]:
    loaded = YOLO(model_path)
    print("Pesos verificados:", model_path, loaded.names)

## 9 — Aciertos, detecciones falsas y objetos perdidos en Validation
Matching por clase, IoU ≥ 0.50, umbral de confianza 0.25. Conteos educativos TP/FP/FN (no sustituye evaluación mAP Ultralytics).

In [ ]:
val_counts = count_detections("models/modelo_elegido.pt", DATA_YAML, split="val",
                              imgsz=IMGSZ, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD)
print(json.dumps(val_counts, indent=2, ensure_ascii=False))
Path("results/validation_counts.json").write_text(json.dumps(val_counts, indent=2, ensure_ascii=False))

## 10 — Test final (SOLO UNA VEZ)
**Ejecutar una única vez** después de la elección. La función crea un marcador para bloquear una segunda evaluación, incluso si el resultado es malo. No usar Test para cambiar hiperparámetros.

In [ ]:
test_metrics = test_once("models/modelo_elegido.pt", DATA_YAML, imgsz=IMGSZ,
                         output="results/test_metrics.json")
print("Resultados TEST — no volver a ejecutar:", json.dumps(test_metrics, indent=2))

## 11 — Análisis cualitativo en Test
La galería permite comparar las etiquetas reales y predicciones del modelo. Seleccionar y comentar **tres fallos**: objeto perdido, falsa detección y caja mal posicionada o clase confundida.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from ultralytics import YOLO
model = YOLO("models/modelo_elegido.pt")
test_images = sorted(p for p in Path("/content/yolo_processed/test/images").rglob("*") if p.suffix.lower() in IMAGE_EXTS)[:6]
fig, axs = plt.subplots(len(test_images), 2, figsize=(12, max(4, 4*len(test_images))))
axs = np.asarray(axs).reshape(-1, 2)
for row, p in zip(axs, test_images):
    from matplotlib.patches import Rectangle
    original = PILImage.open(p).convert("RGB")
    width, height = original.size
    row[0].imshow(original)
    label_file = Path("/content/yolo_processed/test/labels") / p.relative_to(Path("/content/yolo_processed/test/images")).with_suffix(".txt")
    if label_file.exists():
        for ln in label_file.read_text().splitlines():
            cls,x,y,w,h = map(float, ln.split())
            row[0].add_patch(Rectangle(((x-w/2)*width,(y-h/2)*height), w*width,h*height, fill=False,edgecolor="lime",linewidth=2))
            row[0].text((x-w/2)*width,(y-h/2)*height,CLASES[int(cls)],color="black",backgroundcolor="lime",fontsize=7)
    row[0].set_title("Cajas REALES: "+p.name[:28]); row[0].axis("off")
    prediction = model.predict(str(p), imgsz=IMGSZ, conf=CONF_THRESHOLD, verbose=False)[0]
    row[1].imshow(prediction.plot()[:,:,::-1]); row[1].set_title("Predicciones YOLO"); row[1].axis("off")
plt.tight_layout(); plt.savefig("results/test_gallery.png", dpi=140); plt.show()
print("Galería real vs predicho guardada. Seleccionar 3 errores distintos y explicarlos en el PDF.")

### Análisis cualitativo de errores de Test (obligatorio)
Registrar **tres casos reales y diferentes** tras la galería: objeto perdido (FN), falsa detección (FP) y caja mal ubicada o clase confundida. Para cada caso indicar **archivo de Test, clase, comportamiento observado y posible explicación**. Incorporar las capturas a la Sección 5 del PDF; el error de Test **no** debe usarse para cambiar hiperparámetros.

## 12 — Imágenes y videos EXTERNOS al dataset
Usar fotografías y video propios o correctamente citados. No se aceptan los mismos recursos del dataset. Para Evaluación 04 guardar **dos videos diferentes de al menos 10 segundos**.

In [ ]:
from google.colab import files
print("Subir imágenes externas (JPG/PNG), opcional en esta celda")
ext = files.upload()
Path("external/images").mkdir(parents=True, exist_ok=True)
for filename in ext:
    if Path(filename).suffix.lower() in IMAGE_EXTS:
        shutil.move(filename, str(Path("external/images") / Path(filename).name))
external_images = [str(p) for p in Path("external/images").rglob("*") if p.suffix.lower() in IMAGE_EXTS]
if external_images:
    model.predict(source=external_images, imgsz=IMGSZ, conf=CONF_THRESHOLD,
                  save=True, project="/content/YOLO-Maquinaria-UBO/results", name="external_images", exist_ok=True)
else:
    print("SIN IMÁGENES EXTERNAS. Este apartado aún no está cumplido.")

In [ ]:
print("Subir video externo MP4 (archivo corto para Colab)")
vids = files.upload()
Path("external/videos").mkdir(parents=True, exist_ok=True)
for filename in vids:
    if Path(filename).suffix.lower() in {".mp4",".mov",".avi"}:
        shutil.move(filename, str(Path("external/videos") / Path(filename).name))
external_videos = [str(p) for p in Path("external/videos").rglob("*") if p.suffix.lower() in {".mp4",".mov",".avi"}]
import cv2
for filename in external_videos:
    capt = cv2.VideoCapture(filename)
    fps = capt.get(cv2.CAP_PROP_FPS)
    frame_count = capt.get(cv2.CAP_PROP_FRAME_COUNT)
    capt.release()
    if fps <= 0 or frame_count <= 0:
        raise ValueError(f"Video inválido: {filename}")
    duracion = frame_count / fps
    print(f"Video externo: {filename} — {duracion:.1f} s")
    if duracion < 10:
        print("AVISO: este video no cumple el mínimo de 10 segundos exigido para Evaluación 04")
    model.predict(source=filename, imgsz=IMGSZ, conf=CONF_THRESHOLD, save=True,
                  project="/content/YOLO-Maquinaria-UBO/results", name="video_"+Path(filename).stem, exist_ok=True)
if not external_videos:
    print("SIN VIDEO EXTERNO. Este apartado aún no está cumplido.")
if len(external_videos) < 2:
    print("Para Evaluación 04 se requieren DOS videos externos de al menos 10 segundos.")

## 13 — Archivo de resultados y respaldo INMEDIATO
Antes de perder el entorno /content, descargar un ZIP de las evidencias, curvas, notebook aparte y tres pesos entrenados. El dataset no se entrega.

In [ ]:
import datetime
archive = shutil.make_archive("/content/YOLO_E03_resultados", "zip", root_dir=str(ROOT),
     base_dir="results")
with zipfile.ZipFile(archive, "a", zipfile.ZIP_DEFLATED) as z:
    for p in Path("models").glob("*.pt"):
        z.write(p, "models/"+p.name)
print("Archivo listo:", archive)
files.download(archive)
print("IMPORTANTE: Descargar manualmente este notebook con salidas desde Colab: Archivo → Descargar → .ipynb")
print("Entregar también PDF final y pesos .pt. La entrega no incluye el dataset.")